# Capstone Black Box Project

## Function 7 - ML Tuning

## Submission 7 of 13

In [8]:
# Cell 1 - Required imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Matern, WhiteKernel
from scipy.stats import norm
from scipy.stats.qmc import LatinHypercube
from scipy.spatial.distance import cdist
import math

## 1. Load Data

In [9]:
#Cell 2
# Load the .npy file

function = "function_6"

def load_npy_files(function_name):
    initial_inputs = np.load(f'./initial_data/{function_name}/merged_inputs.npy')
    initial_outputs = np.load(f'./initial_data/{function_name}/merged_outputs.npy')
    return initial_inputs, initial_outputs

inputs, outputs = load_npy_files(function)

X=inputs
y=outputs

print(f"n = {len(y)}")
print(inputs)
print(outputs)
print(y)

n = 26
[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.782

## 2. Describe and Review Data

In [10]:
dfi = pd.DataFrame(inputs)
print(dfi.describe())
dfo = pd.DataFrame(outputs)
print(dfo.describe())


               0          1          2          3          4
count  26.000000  26.000000  26.000000  26.000000  26.000000
mean    0.512658   0.480826   0.472537   0.601931   0.357473
std     0.280687   0.301521   0.281467   0.281192   0.290629
min     0.021735   0.001566   0.016523   0.045613   0.004911
25%     0.288376   0.230842   0.317627   0.399721   0.111077
50%     0.525071   0.391958   0.490442   0.676135   0.290546
75%     0.750576   0.782285   0.671909   0.795714   0.596788
max     0.957740   0.931871   0.978806   0.961656   0.892819
               0
count  26.000000
mean   -1.306875
std     0.546179
min    -2.571170
25%    -1.688808
50%    -1.270648
75%    -0.855867
max    -0.413858


## 3. Transform Data

In [11]:
# No transforms applied

y_fit = y

df_y = pd.DataFrame({
    "X1": X[:,0],
    "X2": X[:,1],
    "X3": X[:,2],
    "X4": X[:,3],
    "X5": X[:,4],
    "y": y,
    "y_fit": y_fit
})

output_file = f'./initial_data/{function}/y_transformations.csv'
df_y.to_csv(output_file, index=False)

## 4. Fit Data to Kernal

In [ ]:
# Compare alternative kernels

kernels = {
    'Matern_2.5': ConstantKernel(1.0, (0.01, 10)) * Matern(length_scale=0.3, length_scale_bounds=(0.05, 2.0), nu=2.5),
    'Matern_1.5': ConstantKernel(1.0, (0.01, 10)) * Matern(length_scale=0.3, length_scale_bounds=(0.05, 2.0), nu=1.5),
    'Matern_0.5': ConstantKernel(1.0, (0.01, 10)) * Matern(length_scale=0.3, length_scale_bounds=(0.05, 2.0), nu=0.5),
    # single scalar — isotropic
    'RBF1':        ConstantKernel(1.0, constant_value_bounds=(0.01, 10)) * RBF(length_scale=0.3, length_scale_bounds=(0.05, 2.0)),
    'RBF2':        ConstantKernel(1.0, constant_value_bounds=(0.01, 10)) * RBF(length_scale=0.25, length_scale_bounds=(0.25, 2.0)), 
    'RBF3':        ConstantKernel(1.0, constant_value_bounds=(0.01, 10)) * RBF(length_scale=0.25, length_scale_bounds=(0.01, 10.0)), 
    'RBF4':        ConstantKernel(1.0, (0.01, 10)) * RBF(length_scale=0.25, length_scale_bounds=(0.01, 10.0))
                    + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-5, 1.0)),
    'RBF5':        ConstantKernel(1.0, constant_value_bounds=(0.01, 10)) * RBF(length_scale=np.ones(X.shape[1]),length_scale_bounds=(0.05, 20.0)), 
}

for name, k in kernels.items():
    gpr_test = GaussianProcessRegressor(kernel=k, normalize_y=True, n_restarts_optimizer=20, random_state=42)
    gpr_test.fit(X, y_fit)
    k_fitted = gpr_test.kernel_
    lml = gpr_test.log_marginal_likelihood_value_
    n_params = len(k_fitted.theta)
    aic = -2 * lml + 2 * n_params
    bic = -2 * lml + n_params * np.log(len(y))
    print(f"{name:12s}  LML={lml:.4f}  AIC={aic:.4f}  BIC={bic:.4f}  kernel={k_fitted}")

In [ ]:
# Fit Gaussian Process
# Select the best kernel for fitting

kernel = kernels['RBF5']

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
gpr.fit(X, y_fit)
print(gpr.kernel_)

print(f"LML:       {gpr.log_marginal_likelihood_value_:.4f}")

# Function 6 Customization

In [ ]:
# Wk3 remain unchanged
print("Create kernal for: ", function)
ARD = True

if ARD == False:
    kernel = ConstantKernel(1.0, constant_value_bounds=(0.01, 10.0)) * \
             RBF(length_scale=0.3, # single scalar — isotropic
                 length_scale_bounds=(0.05, 2.0))
else:
    #Wk 5
    kernel = 1.0 * RBF(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(0.05, 2.0)
    )
y_fit = y

# Function 7 Csutomization

In [ ]:
# Wk3 remain unchanged
print("Create kernal for: ", function)
kernel = 1.0 * RBF(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(0.05, 2.0)
)
y_fit = y

# Function 8 Customization

In [ ]:
# Wk3 remain unchanged
#wk4 change the caps
print("Create kernal for: ", function)

kernel = 1.0 * RBF(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(0.05, 20.0)
)
y_fit = y

# Fit Gaussian Process

In [ ]:
# Prediction Grid - Generate candidates and predict y

n_candidates = 10000
low = np.zeros(X.shape[1])
high = np.ones(X.shape[1])

rng = np.random.default_rng(42)
grid = rng.uniform(
low=low,
high=high,
size=(n_candidates, X.shape[1]))

print("Shape:", grid.shape)
print("Dimensions:", grid.ndim)
print("\nFirst 5 rows:")
print(grid[:5])
    
# GP predictions
mu, sigma = gpr.predict(grid, return_std=True)


## 5. Define Acquisition Function

In [ ]:
 
xi_values = {
    "function_4": 0.01, #wk3
    #"function_4": 0.1, #wk2
}

In [ ]:
# Expected Improvement acquisition function
y_best = np.max(y_fit)


xi = xi_values.get(function, 0.1)

print(f"Using xi = {xi} for {function}")

improvement = mu - y_best - xi
Z = improvement / sigma

ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0.0] = 0.0

# Pick next point
next_idx = np.argmax(ei)
print("Next Idx: ", next_idx)
next_x = grid[next_idx]

print("Current best y:", y_best)
print("Next point to evaluate:", next_x)
print("Formatted next point to evaluate:", "-".join(f"{x:.6f}" for x in next_x))
print("Expected improvement:", ei[next_idx])

print("Predicted mean:", mu[next_idx])
print("Predicted std:", sigma[next_idx])
print("Improvement:", mu[next_idx] - y_best - xi)

In [ ]:
# UCB acquisition function

# ---- ACQUISITION: GP-UCB (replaces EI) ----
kappa = 10
ucb = mu + kappa * sigma
 
next_idx = np.argmax(ucb)
 
print(f"\nUsing kappa = {kappa} for function_4")
print("Next Idx: ", next_idx)
print("Current best y:", y_best)
print("Next point to evaluate:", grid[next_idx])
print("UCB value:", ucb[next_idx])
print("Predicted mean:", mu[next_idx])
print("Predicted std:", sigma[next_idx])

In [ ]:
#Function 4 customise

from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P31
r = 0.08  # tight radius — basin is narrow

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(4)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(4)])

sampler = LatinHypercube(d=4, seed=42)
lhs = sampler.random(n=50)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

In [ ]:
#Fuction 6 customise
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P24
r = 0.10

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(5)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(5)])

print("Search bounds:", list(zip(bounds_low, bounds_high)))

sampler = LatinHypercube(d=5, seed=42)
lhs = sampler.random(n=50)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

In [ ]:
#Customise for Function 7
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P34
r = 0.10

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(6)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(6)])

# Override x3 ceiling
bounds_high[2] = 0.30

sampler = LatinHypercube(d=6, seed=42)
lhs = sampler.random(n=50)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

In [ ]:
#Customise function 8
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P44 [0.105, 0.077, 0.174, 0.049, 0.906, 0.632, 0.308, 0.362]
r = 0.10

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(8)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(8)])

# Key constraints
bounds_high[0] = 0.12   # x1 ceiling
bounds_high[2] = 0.15   # x3 ceiling
bounds_low[4]  = 0.90   # x5 floor
bounds_high[6] = 0.25   # x7 ceiling

print("Bounds:", list(zip(bounds_low.round(3), bounds_high.round(3))))

sampler = LatinHypercube(d=8, seed=42)
lhs = sampler.random(n=100)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

## 6. Plots for Uncertainty and Signal Strength 

In [ ]:
# --- Plot ---

